# Lanczos convergence for powers of a rescaled QREM Hamiltonian

This resumable benchmark estimates QREM spectral edges, rescales the operator to approximately `[-1, 1]`, and runs the current `UniPolfed.Numerics.Lanczos` implementation on `H̃ᵖ`. Odd powers target the lower original edge; even powers target states near zero. The committed defaults are a development smoke test.

In [ ]:
import Pkg
const NOTEBOOK_ROOT = normpath(joinpath(@__DIR__, ".."))
Pkg.activate(NOTEBOOK_ROOT)

using UniPolfed
using UniPolfed.Numerics.Lanczos: lanczos, lanczos_extrema, LowestEigs
using CSV, DataFrames, LinearAlgebra, Random, Statistics


## Configuration

Leave `FULL_SWEEP=false` for a quick run. The full setting restores the larger `L=16`, four-power, 100-count, 10-disorder study and can require substantial memory and time. Generated files live under the ignored `notebooks/results/` directory.

In [ ]:
FULL_SWEEP = false
RUN_PLOTS = false
L = FULL_SWEEP ? 16 : 8
D = 1 << L
HX = 1.0
POWERS = FULL_SWEEP ? collect(1:4) : [1, 2]
REQUESTED_COUNTS = FULL_SWEEP ? collect(1:100) : [1, 4, 8]
DISORDERS = FULL_SWEEP ? collect(1:10) : [1]
BASE_SEED = 12_071
MAXDIM = FULL_SWEEP ? 5_000 : D
EXTREMA_MAXDIM = FULL_SWEEP ? 200 : 80
RITZ_TOL = 1e-12
EIGEN_TOL = 1e-9
RESULTS_DIR = joinpath(NOTEBOOK_ROOT, "results")
CSV_PATH = joinpath(RESULTS_DIR, "lanczos_convergence.csv")
PNG_PATH = joinpath(RESULTS_DIR, "lanczos_convergence.png")
PDF_PATH = joinpath(RESULTS_DIR, "lanczos_convergence.pdf")

(; FULL_SWEEP, L, D, POWERS, REQUESTED_COUNTS, DISORDERS, MAXDIM)


## Matrix-free QREM and power maps

Single-spin flips are generated with XOR, so the benchmark does not construct a dense matrix. Workspace for repeated powers is allocated before timing.

In [ ]:
struct QREMMap
    diagonal::Vector{Float64}
    flip_amplitude::Float64
    L::Int
end

function (operator::QREMMap)(y::AbstractVector, x::AbstractVector)
    @. y = operator.diagonal * x
    @inbounds for site in 0:(operator.L - 1)
        mask = 1 << site
        for state in 0:(length(x) - 1)
            y[state + 1] += operator.flip_amplitude * x[(state ⊻ mask) + 1]
        end
    end
    return y
end

make_qrem(L, hx, seed) = QREMMap(
    sqrt(L / 2) .* randn(MersenneTwister(seed), 1 << L), hx / 2, L,
)

struct RescaledMap{F}
    raw!::F
    spread::Float64
    center::Float64
end
function (operator::RescaledMap)(y, x)
    operator.raw!(y, x)
    @. y = (y - operator.center * x) / operator.spread
    return y
end

mutable struct PowerMap{F}
    base!::F
    power::Int
    work1::Any
    work2::Any
end
PowerMap(base!, power) = 1 <= power <= 4 ? PowerMap(base!, power, nothing, nothing) :
    throw(ArgumentError("this notebook supports powers 1:4"))

function (operator::PowerMap)(y, x)
    operator.power == 1 && return operator.base!(y, x)
    if operator.work1 === nothing || size(operator.work1) != size(x)
        operator.work1 = similar(x)
        operator.work2 = similar(x)
    end
    operator.base!(operator.work1, x)
    operator.power == 2 && return operator.base!(y, operator.work1)
    operator.base!(operator.work2, operator.work1)
    operator.power == 3 && return operator.base!(y, operator.work2)
    operator.base!(operator.work1, operator.work2)
    return operator.base!(y, operator.work1)
end


## Benchmark and checkpointing

Each row stores convergence, true eigen-residuals, wall time, and the current named stage timings. A row is skipped only when its `L`, disorder, power, request, and Krylov budget all match.

In [ ]:
load_results(path=CSV_PATH) = isfile(path) ? CSV.read(path, DataFrame) : DataFrame()

function save_results(table, path=CSV_PATH)
    mkpath(dirname(path))
    temporary = path * ".tmp"
    CSV.write(temporary, table)
    mv(temporary, path; force=true)
    return path
end

function already_done(table, disorder, power, requested)
    isempty(table) && return false
    required = (:L, :disorder, :power, :requested, :maxdim)
    all(name -> hasproperty(table, name), required) || return false
    return any(
        (table.L .== L) .& (table.disorder .== disorder) .&
        (table.power .== power) .& (table.requested .== requested) .&
        (table.maxdim .== MAXDIM),
    )
end

function benchmark_one(operator!, value_operator!, x0, disorder, power, requested, Emin, Emax)
    GC.gc(true)
    result = nothing
    elapsed = @elapsed result = lanczos(
        operator!, x0, LowestEigs(requested);
        maxdim=MAXDIM, tol=RITZ_TOL, eigentol=EIGEN_TOL,
        mapvals=value_operator!, collect_garbage=false,
    )
    report = result.report
    max_eigen_residual = isempty(result.residuals) ? Inf : maximum(result.residuals)
    edge_deviation = result.converged == requested ?
        maximum(abs.(collect(result.vals) .+ 1)) : NaN
    return (;
        L, dimension=D, disorder, disorder_seed=BASE_SEED + disorder, power,
        spectral_region=isodd(power) ? "lower_edge" : "near_zero",
        requested, converged=result.converged,
        success=result.converged == requested && report.complete,
        iterations=report.itersneeded, krylov_columns=report.ncols_cpu_needed,
        convergence_checks=report.numofchecks, elapsed_seconds=elapsed,
        factorization_seconds=UniPolfed.Common.total_wall(report.timer),
        mapping_seconds=report.timer[:matvec],
        max_eigen_residual, max_deviation_from_minus_one=edge_deviation,
        raw_Emin=Emin, raw_Emax=Emax,
        rescale_center=(Emax + Emin) / 2, rescale_halfwidth=(Emax - Emin) / 2,
        maxdim=MAXDIM, ritz_tol=RITZ_TOL, eigen_tol=EIGEN_TOL,
    )
end


In [ ]:
results = load_results()
for disorder in DISORDERS
    raw_operator = make_qrem(L, HX, BASE_SEED + disorder)
    raw! = (y, x) -> raw_operator(y, x)
    bound_seed = randn(MersenneTwister(BASE_SEED + 100_000 + disorder), D)
    Emin, Emax = lanczos_extrema(
        raw!, bound_seed; maxdim=EXTREMA_MAXDIM, tol=RITZ_TOL,
        eigentol=EIGEN_TOL, power_iters=3,
    )
    spread, center = (Emax - Emin) / 2, (Emax + Emin) / 2
    @assert spread > 0
    scaled_operator = RescaledMap(raw!, spread, center)
    scaled! = (y, x) -> scaled_operator(y, x)
    println("disorder=$disorder: estimated raw spectrum [$Emin, $Emax]")

    for power in POWERS
        power_operator = PowerMap(scaled!, power)
        operator! = (y, x) -> power_operator(y, x)
        warm_x = randn(MersenneTwister(BASE_SEED + 200_000 + 10disorder + power), D)
        operator!(similar(warm_x), warm_x)

        for requested in REQUESTED_COUNTS
            already_done(results, disorder, power, requested) && continue
            seed = BASE_SEED + 1_000_000disorder + 10_000power + requested
            x0 = randn(MersenneTwister(seed), D)
            row = benchmark_one(operator!, scaled!, x0, disorder, power, requested, Emin, Emax)
            push!(results, row; cols=:union)
            sort!(results, [:L, :disorder, :power, :requested])
            save_results(results)
            println("  p=$power n=$requested: $(row.iterations) iterations, " *
                    "$(round(row.elapsed_seconds; digits=3)) s, converged=$(row.converged)")
        end
    end
end
results


## Aggregate and optionally plot

Smoke results are useful for regression detection, not hardware claims. Full-sweep lines average disorder realizations and use one-standard-deviation ribbons.

In [ ]:
summary = combine(
    groupby(results[results.L .== L, :], [:power, :requested]),
    :iterations => mean => :iterations_mean,
    :iterations => (values -> length(values) > 1 ? std(values) : 0.0) => :iterations_std,
    :elapsed_seconds => mean => :time_mean,
    :elapsed_seconds => (values -> length(values) > 1 ? std(values) : 0.0) => :time_std,
    :success => mean => :success_fraction,
    nrow => :samples,
)
summary


In [ ]:
if RUN_PLOTS
    using PythonPlot
    mkpath(RESULTS_DIR)
    figure, axes = subplots(2, 1; figsize=(10, 8), constrained_layout=true)
    for power in POWERS
        part = sort(summary[summary.power .== power, :], :requested)
        isempty(part) && continue
        axes[0].plot(part.requested, part.iterations_mean; marker="o", label="H̃^$power")
        axes[1].plot(part.requested, part.time_mean; marker="o", label="H̃^$power")
    end
    axes[0].set(xlabel="requested eigenpairs", ylabel="iterations", title="QREM L=$L convergence")
    axes[1].set(xlabel="requested eigenpairs", ylabel="seconds", title="wall time")
    foreach(axis -> (axis.grid(true; alpha=0.25); axis.legend()), axes)
    figure.savefig(PNG_PATH; dpi=180)
    figure.savefig(PDF_PATH)
    figure
end
